In [1]:
import numpy as np
import tensorflow as tf
import jax
import jax.numpy as jnp
import flax.linen as nn
from flax import jax_utils
import optax
from flax.training import train_state
import time
from jax.sharding import PositionalSharding

from Preprocess import prepare_tcn_data
from TCN_model import TCN, init_train_state, train_step

I0000 00:00:1776672311.145674  140131 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776672311.146892  140131 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776672311.197873  140131 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776672312.574423  140131 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONE

In [2]:
with tf.io.gfile.GFile("gs://fi2010-lob-data/BenchmarkDatasets/NoAuction/1.NoAuction_Zscore/NoAuction_Zscore_Training/Train_Dst_NoAuction_ZScore_CF_1.txt", 'r') as f:
    raw_data = np.loadtxt(f)

## Test: repeated data pipeline + jax.jit

### batch size: 1024

In [3]:
batch_size = 1024
time_step = 127

In [4]:
# Setting sharding strategy
num_devices = jax.local_device_count() 
sharding = PositionalSharding(jax.devices())
print(f'Number of TPU devices: {num_devices}.')

Number of TPU devices: 4.


In [5]:
# How you cut the data
x_sharding = sharding.reshape(num_devices, 1, 1)
y_sharding = sharding.reshape(num_devices)

In [6]:
train_dataset = prepare_tcn_data(raw_data, batch_size, time_step, True, True)
train_iter = train_dataset.as_numpy_iterator()

E0000 00:00:1776672325.383358  140131 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [7]:
rng = jax.random.PRNGKey(3721)
rng, dropout_rng = jax.random.split(rng)
model = TCN()
state, dropout_rng = init_train_state(rng, model)
# We don't need next two lines because its now control by XLA
# dropout_rngs = jax.random.split(dropout_rng, 4)
# replicated_state = jax_utils.replicate(state)

In [8]:
# Data warmup
next_batch = next(train_iter)

# No reshape. The dispatch is controlled by XLA
parallel_x = next_batch[0].astype(jnp.bfloat16)
parallel_y = next_batch[1].astype(jnp.int32)

# Sending data to TPU HBM by the sharding strategy
parallel_x = jax.device_put(parallel_x, x_sharding)
parallel_y = jax.device_put(parallel_y, y_sharding)

In [9]:
# Because the data is going to be repeated, we need set a total step
# And there is no more Epoch.
TOTAL_STEPS = 500 
start_time = time.time()
running_loss = 0.0
XLA_time = 0.0

print(f"--- Experiment 2: Infinite Stream & Auto-Sharding ---")

for step in range(TOTAL_STEPS):
    # Calculation
    state, loss, acc = train_step(state, parallel_x, parallel_y, dropout_rng)
    
    # Asynchronous extraction
    next_batch = next(train_iter)
    cpu_x = next_batch[0].astype(jnp.bfloat16)
    cpu_y = next_batch[1].astype(jnp.int32)
    
    parallel_x = jax.device_put(cpu_x, x_sharding)
    parallel_y = jax.device_put(cpu_y, y_sharding)
    
    # Returning Single Number
    running_loss += loss.item()

    if step == 0:
        XLA_time = time.time() - start_time
    if step == 1:
        XLA_time_2 = time.time() - start_time - XLA_time
    if step % 50 == 0:
        print(f" Step {step:04d} | 实时 Loss: {loss.item():.4f} | Acc: {acc.item():.4f}")

total_time = time.time() - start_time
print(f"✅ Done {TOTAL_STEPS} Steps! Total time: {total_time:.2f}s")
print(f'🚀 XLA Time + first step(approximate): {XLA_time}.')
print(f'🚀 interesting: {XLA_time_2}.')
step_per_sec = (total_time-XLA_time-XLA_time_2)/(TOTAL_STEPS - 2)
print(f'⚡ Second per step: {step_per_sec:.8f}s ')
print(f"📊 Data I/O: {(1/step_per_sec)*batch_size:.0f} samples/sec")

--- Experiment 2: Infinite Stream & Auto-Sharding ---
 Step 0000 | 实时 Loss: 9.1505 | Acc: 0.1387
 Step 0050 | 实时 Loss: 1.2000 | Acc: 0.3818
 Step 0100 | 实时 Loss: 1.0651 | Acc: 0.4385
 Step 0150 | 实时 Loss: 1.0322 | Acc: 0.4580
 Step 0200 | 实时 Loss: 0.9298 | Acc: 0.5479
 Step 0250 | 实时 Loss: 0.9022 | Acc: 0.5479
 Step 0300 | 实时 Loss: 0.9758 | Acc: 0.5068
 Step 0350 | 实时 Loss: 0.8562 | Acc: 0.6035
 Step 0400 | 实时 Loss: 0.7618 | Acc: 0.6572
 Step 0450 | 实时 Loss: 0.9298 | Acc: 0.5498
✅ Done 500 Steps! Total time: 84.05s
🚀 XLA Time + first step(approximate): 37.802526235580444.
🚀 interesting: 37.0838565826416.
⚡ Second per step: 0.01840203s 
📊 Data I/O: 55646 samples/sec


### batch size: 4096

In [10]:
batch_size = 4096
time_step = 127

In [11]:
train_dataset = prepare_tcn_data(raw_data, batch_size, time_step, True, True)
train_iter = train_dataset.as_numpy_iterator()

In [12]:
rng = jax.random.PRNGKey(3721)
rng, dropout_rng = jax.random.split(rng)
model = TCN()
state, dropout_rng = init_train_state(rng, model)

In [13]:
# Data warmup
next_batch = next(train_iter)

# No reshape. The dispatch is controlled by XLA
parallel_x = next_batch[0].astype(jnp.bfloat16)
parallel_y = next_batch[1].astype(jnp.int32)

# Sending data to TPU HBM by the sharding strategy
parallel_x = jax.device_put(parallel_x, x_sharding)
parallel_y = jax.device_put(parallel_y, y_sharding)

In [14]:
TOTAL_STEPS = 500 
start_time = time.time()
running_loss = 0.0
XLA_time = 0.0

print(f"--- Experiment 2: Infinite Stream & Auto-Sharding ---")

for step in range(TOTAL_STEPS):
    # Calculation
    state, loss, acc = train_step(state, parallel_x, parallel_y, dropout_rng)
    
    # Asynchronous extraction
    next_batch = next(train_iter)
    cpu_x = next_batch[0].astype(jnp.bfloat16)
    cpu_y = next_batch[1].astype(jnp.int32)
    
    parallel_x = jax.device_put(cpu_x, x_sharding)
    parallel_y = jax.device_put(cpu_y, y_sharding)
    
    # Returning Single Number
    running_loss += loss.item()

    if step == 0:
        XLA_time = time.time() - start_time
    if step == 1:
        XLA_time_2 = time.time() - start_time - XLA_time
    if step % 50 == 0:
        print(f" Step {step:04d} | 实时 Loss: {loss.item():.4f} | Acc: {acc.item():.4f}")

total_time = time.time() - start_time
print(f"✅ Done {TOTAL_STEPS} Steps! Total time: {total_time:.2f}s")
print(f'🚀 XLA Time + first step(approximate): {XLA_time}.')
print(f'🚀 interesting: {XLA_time_2}.')
step_per_sec = (total_time-XLA_time-XLA_time_2)/(TOTAL_STEPS - 2)
print(f'⚡ Sconed per step: {step_per_sec:.8f}s ')
print(f"📊 Data I/O: {(1/step_per_sec)*batch_size:.0f} samples/sec")

--- Experiment 2: Infinite Stream & Auto-Sharding ---
 Step 0000 | 实时 Loss: 8.9649 | Acc: 0.1460
 Step 0050 | 实时 Loss: 1.1558 | Acc: 0.4519
 Step 0100 | 实时 Loss: 1.0095 | Acc: 0.4944
 Step 0150 | 实时 Loss: 1.0135 | Acc: 0.4658
 Step 0200 | 实时 Loss: 0.9937 | Acc: 0.4858
 Step 0250 | 实时 Loss: 0.9021 | Acc: 0.5720
 Step 0300 | 实时 Loss: 0.8497 | Acc: 0.6016
 Step 0350 | 实时 Loss: 0.8108 | Acc: 0.6208
 Step 0400 | 实时 Loss: 0.9346 | Acc: 0.5154
 Step 0450 | 实时 Loss: 0.9577 | Acc: 0.5107
✅ Done 500 Steps! Total time: 96.62s
🚀 XLA Time + first step(approximate): 28.496290922164917.
🚀 interesting: 29.468626260757446.
⚡ Sconed per step: 0.07762980s 
📊 Data I/O: 52763 samples/sec
